# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
%pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import random
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [4]:
import numpy as np
from sklearn.datasets import load_digits

def load_mnist_via_digits(num_training=1400, num_validation=200, num_test=197, random_state=42):
    """
    Загружает датасет digits из sklearn (аналог 'маленького MNIST')
    и подготавливает его примерно так же, как CIFAR-10:
    - загрузка
    - перемешивание
    - разбиение на train / val / test
    - нормализация по train
    
    Возвращает:
    X_train, y_train, X_val, y_val, X_test, y_test
    """
    digits = load_digits()

    # images: (N, 8, 8), target: (N,)
    X = np.asarray(digits.images, dtype=np.float32)
    y = np.asarray(digits.target, dtype=np.int32)

    # Перемешивание
    rng = np.random.RandomState(random_state)
    mask = np.arange(X.shape[0])
    rng.shuffle(mask)

    X = X[mask]
    y = y[mask]

    # Разбиение
    X_train = X[:num_training]
    y_train = y[:num_training]

    X_val = X[num_training:num_training + num_validation]
    y_val = y[num_training:num_training + num_validation]

    X_test = X[num_training + num_validation:num_training + num_validation + num_test]
    y_test = y[num_training + num_validation:num_training + num_validation + num_test]

    # Нормализация по train
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    std_pixel[std_pixel == 0] = 1.0

    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist_via_digits()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (1400, 8, 8)
Train labels shape:  (1400,) int32
Validation data shape:  (200, 8, 8)
Validation labels shape:  (200,)
Test data shape:  (197, 8, 8)
Test labels shape:  (197,)


In [5]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[idxs[i:i+B]], self.y[idxs[i:i+B]]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [6]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 8, 8) (64,)
1 (64, 8, 8) (64,)
2 (64, 8, 8) (64,)
3 (64, 8, 8) (64,)
4 (64, 8, 8) (64,)
5 (64, 8, 8) (64,)
6 (64, 8, 8) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [7]:
device = '/GPU:0'

In [8]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [9]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            kernel_initializer=initializer
        )
        self.relu1 = tf.keras.layers.ReLU()

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer
        )
        self.relu2 = tf.keras.layers.ReLU()

        self.flatten = tf.keras.layers.Flatten()

        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(x.shape) == 3:
            x = tf.expand_dims(x, axis=-1)

        x = self.conv1(x)
        x = self.relu1(x)

        x = self.conv2(x)
        x = self.relu2(x)

        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [10]:
def test_ThreeLayerConvNet():
    x = tf.zeros((64, 8, 8))
    model = ThreeLayerConvNet(channel_1=32, channel_2=16, num_classes=10)
    scores = model(x)
    print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [11]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [12]:
hidden_size, num_classes = 100, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

print_every = 10
train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.8238110542297363, Accuracy: 15.625, Val Loss: 2.926320791244507, Val Accuracy: 17.0
Iteration 10, Epoch 1, Loss: 2.721428871154785, Accuracy: 16.051136016845703, Val Loss: 2.52473783493042, Val Accuracy: 25.0
Iteration 20, Epoch 1, Loss: 2.540686845779419, Accuracy: 20.4613094329834, Val Loss: 2.2644057273864746, Val Accuracy: 31.5


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [13]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.7361109256744385, Accuracy: 7.8125, Val Loss: 3.0110535621643066, Val Accuracy: 10.5
Iteration 10, Epoch 1, Loss: 2.3419289588928223, Accuracy: 25.99431800842285, Val Loss: 1.3174759149551392, Val Accuracy: 72.0
Iteration 20, Epoch 1, Loss: 1.6856404542922974, Accuracy: 50.967262268066406, Val Loss: 0.6251212358474731, Val Accuracy: 88.0


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [14]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (8, 8)
    hidden_layer_size, num_classes = 128, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.1229844093322754, Accuracy: 6.25, Val Loss: 2.970972776412964, Val Accuracy: 4.0
Iteration 10, Epoch 1, Loss: 2.7023019790649414, Accuracy: 5.965909004211426, Val Loss: 2.408330202102661, Val Accuracy: 6.0
Iteration 20, Epoch 1, Loss: 2.4583206176757812, Accuracy: 11.160714149475098, Val Loss: 2.0640995502471924, Val Accuracy: 21.0


/Users/daschaiva/Library/Python/3.9/lib/python/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Альтернативный менее гибкий способ обучения:

In [15]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.3220 - sparse_categorical_accuracy: 0.2181 - val_loss: 1.6384 - val_sparse_categorical_accuracy: 0.5450
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.6639 - sparse_categorical_accuracy: 0.4372


[1.6417769193649292, 0.46192893385887146]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [16]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    input_shape = (8, 8)
    channel_1, channel_2, num_classes = 32, 16, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Reshape((8, 8, 1)),
        tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),
        tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.715116262435913, Accuracy: 7.8125, Val Loss: 2.259962797164917, Val Accuracy: 17.5
Iteration 10, Epoch 1, Loss: 1.9893913269042969, Accuracy: 28.551137924194336, Val Loss: 1.478263258934021, Val Accuracy: 55.0
Iteration 20, Epoch 1, Loss: 1.604612112045288, Accuracy: 48.511905670166016, Val Loss: 0.8795087337493896, Val Accuracy: 80.5


In [17]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.1782 - sparse_categorical_accuracy: 0.2872 - val_loss: 1.1637 - val_sparse_categorical_accuracy: 0.7600
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2242 - sparse_categorical_accuracy: 0.7338


[1.1933075189590454, 0.7512690424919128]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [18]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [19]:
input_shape = (8, 8)
hidden_size, num_classes = 128, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.855355739593506, Accuracy: 6.25, Val Loss: 2.9202942848205566, Val Accuracy: 14.5
Iteration 10, Epoch 1, Loss: 2.395676374435425, Accuracy: 16.903409957885742, Val Loss: 2.420604944229126, Val Accuracy: 17.0
Iteration 20, Epoch 1, Loss: 2.2080235481262207, Accuracy: 22.693452835083008, Val Loss: 2.1051511764526367, Val Accuracy: 27.000001907348633


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [20]:
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist_via_digits()

X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(1400, 8, 8, 1)
(200, 8, 8, 1)
(197, 8, 8, 1)


In [21]:
batch_size = 64
train_dset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train)).batch(batch_size)
val_dset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)
test_dset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)

In [22]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer
        )
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(
            filters=64,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer
        )
        self.bn2 = tf.keras.layers.BatchNormalization()

        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=2, strides=2)
        self.dropout1 = tf.keras.layers.Dropout(0.25)

        self.conv3 = tf.keras.layers.Conv2D(
            filters=128,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer
        )
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.flatten = tf.keras.layers.Flatten()

        self.fc1 = tf.keras.layers.Dense(
            128,
            kernel_initializer=initializer
        )
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.dropout2 = tf.keras.layers.Dropout(0.5)

        self.fc2 = tf.keras.layers.Dense(
            10,
            activation='softmax',
            kernel_initializer=initializer
        )

        self.relu = tf.keras.layers.ReLU()

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.relu(x)

        x = self.pool1(x)
        x = self.dropout1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = self.relu(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.bn4(x, training=training)
        x = self.relu(x)
        x = self.dropout2(x, training=training)

        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 20
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

2026-04-16 04:01:07.143719: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Iteration 0, Epoch 1, Loss: 3.2082557678222656, Accuracy: 6.25, Val Loss: 2.828465461730957, Val Accuracy: 23.5


2026-04-16 04:01:07.927650: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Iteration 20, Epoch 1, Loss: 1.080840826034546, Accuracy: 65.32738494873047, Val Loss: 0.5328128933906555, Val Accuracy: 80.5


2026-04-16 04:01:08.758837: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Iteration 40, Epoch 2, Loss: 0.3261226415634155, Accuracy: 92.10526275634766, Val Loss: 0.18436390161514282, Val Accuracy: 93.5
Iteration 60, Epoch 3, Loss: 0.22138966619968414, Accuracy: 94.30146789550781, Val Loss: 0.15159548819065094, Val Accuracy: 94.0


2026-04-16 04:01:10.348159: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Iteration 80, Epoch 4, Loss: 0.16611218452453613, Accuracy: 96.35417175292969, Val Loss: 0.10319676995277405, Val Accuracy: 96.5
Iteration 100, Epoch 5, Loss: 0.12279052287340164, Accuracy: 97.95672607421875, Val Loss: 0.07668959349393845, Val Accuracy: 97.5
Iteration 120, Epoch 6, Loss: 0.08641651272773743, Accuracy: 98.57954406738281, Val Loss: 0.06819278746843338, Val Accuracy: 98.0
Iteration 140, Epoch 7, Loss: 0.08877221494913101, Accuracy: 98.2638931274414, Val Loss: 0.057299770414829254, Val Accuracy: 99.0


2026-04-16 04:01:13.559642: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Iteration 160, Epoch 8, Loss: 0.07568752765655518, Accuracy: 98.4375, Val Loss: 0.06145131587982178, Val Accuracy: 99.0
Iteration 180, Epoch 9, Loss: 0.062180303037166595, Accuracy: 98.75, Val Loss: 0.03807825595140457, Val Accuracy: 99.5
Iteration 200, Epoch 10, Loss: 0.051923107355833054, Accuracy: 99.47917175292969, Val Loss: 0.030805783346295357, Val Accuracy: 99.5


In [23]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        reg = tf.keras.regularizers.l2(1e-4)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer,
            kernel_regularizer=reg
        )
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer,
            kernel_regularizer=reg
        )
        self.bn2 = tf.keras.layers.BatchNormalization()

        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=2, strides=2)
        self.dropout1 = tf.keras.layers.Dropout(0.2)

        self.conv3 = tf.keras.layers.Conv2D(
            filters=64,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer,
            kernel_regularizer=reg
        )
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.conv4 = tf.keras.layers.Conv2D(
            filters=64,
            kernel_size=3,
            padding='same',
            kernel_initializer=initializer,
            kernel_regularizer=reg
        )
        self.bn4 = tf.keras.layers.BatchNormalization()

        self.gap = tf.keras.layers.GlobalAveragePooling2D()

        self.fc1 = tf.keras.layers.Dense(
            64,
            kernel_initializer=initializer,
            kernel_regularizer=reg
        )
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.dropout2 = tf.keras.layers.Dropout(0.3)

        self.fc2 = tf.keras.layers.Dense(
            10,
            activation='softmax',
            kernel_initializer=initializer
        )

        self.relu = tf.keras.layers.ReLU()

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(input_tensor.shape) == 3:
            input_tensor = tf.expand_dims(input_tensor, axis=-1)

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.relu(x)

        x = self.pool1(x)
        x = self.dropout1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = self.relu(x)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = self.relu(x)

        x = self.gap(x)

        x = self.fc1(x)
        x = self.bn5(x, training=training)
        x = self.relu(x)
        x = self.dropout2(x, training=training)

        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 20
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 5e-4
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.0843122005462646, Accuracy: 4.6875, Val Loss: 3.9708938598632812, Val Accuracy: 10.0
Iteration 20, Epoch 1, Loss: 2.1442532539367676, Accuracy: 30.05952262878418, Val Loss: 3.113311290740967, Val Accuracy: 8.5
Iteration 40, Epoch 2, Loss: 1.2699530124664307, Accuracy: 59.21052551269531, Val Loss: 2.3250062465667725, Val Accuracy: 14.0
Iteration 60, Epoch 3, Loss: 0.9170706868171692, Accuracy: 75.45955657958984, Val Loss: 1.9300191402435303, Val Accuracy: 33.0
Iteration 80, Epoch 4, Loss: 0.6560580730438232, Accuracy: 85.83333587646484, Val Loss: 1.5799918174743652, Val Accuracy: 61.0
Iteration 100, Epoch 5, Loss: 0.5306205153465271, Accuracy: 88.34134674072266, Val Loss: 1.2171862125396729, Val Accuracy: 86.0


2026-04-16 04:01:20.355824: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Iteration 120, Epoch 6, Loss: 0.4180678427219391, Accuracy: 93.0397720336914, Val Loss: 1.0370676517486572, Val Accuracy: 89.0
Iteration 140, Epoch 7, Loss: 0.3451773524284363, Accuracy: 95.1388931274414, Val Loss: 0.8446446061134338, Val Accuracy: 91.0
Iteration 160, Epoch 8, Loss: 0.3083074688911438, Accuracy: 95.98213958740234, Val Loss: 0.6497279405593872, Val Accuracy: 93.0
Iteration 180, Epoch 9, Loss: 0.2727814316749573, Accuracy: 95.0, Val Loss: 0.5261558890342712, Val Accuracy: 94.0
Iteration 200, Epoch 10, Loss: 0.19750452041625977, Accuracy: 97.39582824707031, Val Loss: 0.4189065992832184, Val Accuracy: 95.5


Опишите все эксперименты, результаты. Сделайте выводы.

В первом эксперименте была использована более глубокая сверточная архитектура: сначала слой Conv2D на 32 фильтра с ядром 3x3, затем Batch Normalization и ReLU, после этого второй Conv2D на 64 фильтра с ядром 3x3, снова Batch Normalization и ReLU, далее MaxPooling 2x2 и Dropout 0.25. После этого добавлялся третий сверточный слой Conv2D на 128 фильтров с ядром 3x3, затем Batch Normalization и ReLU, а в конце шли Flatten, полносвязный слой на 128 нейронов, Batch Normalization, ReLU, Dropout 0.5 и выходной Dense на 10 классов с softmax. Для обучения использовался оптимизатор Adam с learning rate 1e-3. Этот вариант показал лучший результат из двух: на первой эпохе точность на валидации выросла до 80.5%, на 5-й эпохе достигла 97.5%, на 8-й эпохе составила 99.0%, на 9-й эпохе тоже была 99.0%, а после 10-й эпохи осталась на очень высоком уровне 99.5%. По этому эксперименту можно сделать вывод, что сочетание более глубокой архитектуры, Batch Normalization, Dropout и Adam очень хорошо подошло для задачи классификации изображений digits и обеспечило быструю и устойчивую сходимость.

Во втором эксперименте была проверена другая архитектура с более сильной регуляризацией. В ней использовались два первых сверточных слоя Conv2D на 32 фильтра, затем Batch Normalization, ReLU, MaxPooling 2x2 и Dropout 0.2, после этого еще два сверточных слоя Conv2D на 64 фильтра с Batch Normalization и ReLU, затем вместо Flatten применялся слой GlobalAveragePooling2D. После него шли полносвязный слой на 64 нейрона, Batch Normalization, ReLU, Dropout 0.3 и выходной Dense на 10 классов с softmax. Для сверточных и полносвязного скрытого слоя также была добавлена L2-регуляризация, а обучение выполнялось с помощью Adam с learning rate 5e-4. Эта модель тоже дала хороший результат, но обучалась заметно медленнее: на первой эпохе точность на валидации была только 8.5%, на 4-й эпохе выросла до 61.0%, на 7-й эпохе достигла 91.0%, а после 10-й эпохи составила 95.5%. Значит, второй эксперимент тоже успешно решил задачу и превысил требуемый порог, но по итоговой точности и скорости обучения уступил первому варианту, поэтому наиболее удачной архитектурой из двух оказалась первая.
